In [1]:
# controle van mijn bestanden 

from pathlib import Path
import os

print("Beschikbare datasets:")
print(os.listdir("/kaggle/input"))

print("\nZoeken naar HMS-train.csv:")
for file in Path("/kaggle/input").rglob("train.csv"):
    print(file)

print("\nZoeken naar het baseline-script:")
for file in Path("/kaggle/input").rglob("hms_sklearn_baseline.py"):
    print(file)

Beschikbare datasets:
['datasets', 'competitions']

Zoeken naar HMS-train.csv:
/kaggle/input/competitions/hms-harmful-brain-activity-classification/train.csv

Zoeken naar het baseline-script:
/kaggle/input/datasets/hannaleysen/github-link-exp-6/src/hms_sklearn_baseline.py


In [2]:
#bestaande baseline hergebruiken ! enkel de dingen veranderen die je meot veranderen
# Eerst de functies uit de baseline eventjes controleren

from pathlib import Path
import importlib.util
import inspect

# Python-Bestand zoeken in GitHub-dataset
scripts = list(
    Path("/kaggle/input").rglob("hms_sklearn_baseline.py")
)

if len(scripts) != 1:
    raise FileNotFoundError(
        f"Verwacht één baselinebestand, gevonden: {scripts}"
    )

SCRIPT_PATH = scripts[0]
print("Baseline gevonden op:", SCRIPT_PATH)
# Importeer bestaande baseline als Python-module
spec = importlib.util.spec_from_file_location(
    "hms_baseline",
    SCRIPT_PATH
)

baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)

print("Baseline succesvol geïmporteerd!")
print("\nBeschikbare functies:")

for name in [
    "load_labels",
    "build_features",
    "kl_score",
    "cross_validate",
    "make_model",
    "to_matrix",
    "expand"
]:
    function = getattr(baseline, name, None)
    if function is not None:
        print(name, inspect.signature(function))


Baseline gevonden op: /kaggle/input/datasets/hannaleysen/github-link-exp-6/src/hms_sklearn_baseline.py
Baseline succesvol geïmporteerd!

Beschikbare functies:
load_labels (data_dir, max_specs=None, seed=42)
build_features (meta, spec_dir, is_test=False, n_jobs=-1)
kl_score (y_true, y_pred, eps=1e-15)
cross_validate (X, Y, groups, num_votes, n_folds, seed)
make_model (name, seed=42)
to_matrix (frame, feats)
expand (X, Y)


In [3]:
# KNN --> de 6 expertkansen rechtstreeks gebruiken en niet eerst omzetten naar gewone klassevoorbeelden 

import inspect

print("Informatie over build_features:")
print(inspect.getsource(baseline.build_features))

print("\nInformatie over cross_validate:")
print(inspect.getsource(baseline.cross_validate))


Informatie over build_features:
def build_features(meta, spec_dir, is_test=False, n_jobs=-1):
    """Compute features for all EEGs in `meta`, reading each spectrogram file once (in parallel)."""
    t0 = time.time()
    if is_test:   # test.csv has no label offsets
        meta = meta.assign(off_min=0.0, off_max=0.0)
    jobs = [delayed(spectrogram_features)(spec_dir / f"{sid}.parquet",
                                          list(zip(g.eeg_id, g.off_min, g.off_max)), is_test)
            for sid, g in meta.groupby("spectrogram_id")]
    results = Parallel(n_jobs=n_jobs, batch_size=32)(jobs)
    out = pd.DataFrame([row for rows in results for row in rows])
    print(f"{len(jobs)} spectrograms -> {len(out)} feature rows in {time.time() - t0:.0f}s")
    return out


Informatie over cross_validate:
def cross_validate(X, Y, groups, num_votes, n_folds, seed):
    """Patient-grouped K-fold CV for both models; prints scores and returns the better model's name."""
    prior = np.tile(Y.mean(

In [4]:
# KNN-experiment 


In [5]:
# eerst nog dingetjes ophalen uit de baseline 
import inspect

print("MODEL_NAMES:")
print(baseline.MODEL_NAMES)              # welke modellen vergelijkt de baseline momenteel

print("\nFunctie fit:")
print(inspect.getsource(baseline.fit))   # Hoe worden bestaande modellen getraind 

print("\nFunctie load_labels:")
print(inspect.getsource(baseline.load_labels))  # hoe worden de 6 doelklassen ingelezen

print("\nFunctie to_matrix:")
print(inspect.getsource(baseline.to_matrix))    # om exact dezelfde featureselectie als de baseline te gebruiken

MODEL_NAMES:
['logreg', 'hgb']

Functie fit:
def fit(name, X, Y, seed=42):
    """Fit model `name` on features X and soft labels Y (rows are vote distributions)."""
    Xe, ye, we = expand(X, Y)
    m = make_model(name, seed)
    if name == "logreg":   # in a Pipeline the weight must be routed to the final step
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m


Functie load_labels:
def load_labels(data_dir, max_specs=None, seed=42):
    """Return one row per EEG with its spectrogram id, label offsets, patient id and vote distribution."""
    train = pd.read_csv(data_dir / "train.csv")

    # train.csv has several labelled sub-segments per eeg_id: sum their votes and keep the offset range.
    agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"], "patient_id": "first"}
    agg.update({c: "sum" for c in CLASSES})
    df = train.groupby("eeg_id").agg(agg)
    df.columns = ["spectrogra

In [6]:
# Hoe brengt de baseline alles samen?  --> ervoor zorgen dat KNN dezelfde invoergegevens krijgt als de andere 2 modellen in de baseline 
import inspect

print("1. De functie main:")  
print(inspect.getsource(baseline.main))   # volgorde dataverwerking baseline (gehele workflow)

print("\n2. De functie make_model:")
print(inspect.getsource(baseline.make_model))   # hoe modellen instellen 

print("\n3. De beschikbare klassen:")
print(baseline.CLASSES)                   # volgorde doelklassen bevestigen (KNN moet deze ook hebben)

1. De functie main:
def main():
    parser = argparse.ArgumentParser(description="HMS simple scikit-learn baseline")
    parser.add_argument("--data-dir", type=Path, required=True,
                        help="folder that contains train.csv, test.csv, train_spectrograms/ and test_spectrograms/")
    parser.add_argument("--mode", choices=["full", "quick"], default="full",
                        help="full: all training spectrograms; quick: a random 3,000 for a fast check")
    parser.add_argument("--no-cv", action="store_true", help="skip cross-validation and use gradient boosting")
    parser.add_argument("--folds", type=int, default=5, help="number of cross-validation folds")
    parser.add_argument("--n-jobs", type=int, default=-1, help="CPU cores for reading spectrograms (-1 = all)")
    parser.add_argument("--output", type=Path, default=Path("submission.csv"), help="where to write predictions")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

In [7]:
# data inladen en features bouwen (zelfde gebruiken als baseline)
from pathlib import Path
import numpy as np
import pandas as pd

# Pad naar de HMS-competitiedataset
DATA_DIR = Path(
    "/kaggle/input/competitions/hms-harmful-brain-activity-classification"
)

# Controleer of de dataset gevonden wordt
assert (DATA_DIR / "train.csv").exists(), "train.csv niet gevonden!"

# 1. Labels inladen
df = baseline.load_labels(
    DATA_DIR,
    max_specs=3000,
    seed=42
)

print(df.shape, "EEGs")
print(df.patient_id.nunique(), "verschillende patiënten")
print(df.spectrogram_id.nunique(), "spectrogrammen")

# 2. Features berekenen
X_df = baseline.build_features(
    df,
    DATA_DIR / "train_spectrograms",
    n_jobs=2
)

# 3. Labels en features samenvoegen
df = df.merge(X_df, on="eeg_id")

# 4. Dezelfde featureselectie gebruiken als de baseline
feats = [c for c in X_df.columns if c != "eeg_id"]

X = baseline.to_matrix(df, feats)
Y = df[baseline.CLASSES].to_numpy(dtype=np.float64)

groups = df["patient_id"].to_numpy()
num_votes = df["num_votes"].to_numpy()

print("\nData klaar voor KNN!")
print("Aantal EEG-records:", len(X))
print("Aantal features:", X.shape[1])
print("Aantal klassen:", Y.shape[1])
print("Vorm van X:", X.shape)
print("Vorm van Y:", Y.shape)

(4738, 12) EEGs
1163 verschillende patiënten
3000 spectrogrammen
3000 spectrograms -> 4738 feature rows in 82s

Data klaar voor KNN!
Aantal EEG-records: 4738
Aantal features: 88
Aantal klassen: 6
Vorm van X: (4738, 88)
Vorm van Y: (4738, 6)


In [8]:
# Bibliotheken importeren 
import numpy as np
import pandas as pd
import torch              # neurale netwerk bouwen & trainen
import torch.nn as nn     # bouwstenen voor neurale netwerken

from torch.utils.data import TensorDataset, DataLoader  # trainingsgegevens verdelen 
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

In [9]:
# MLP model bouwen

class MLPClassifier(nn.Module):

    def __init__(self, input_size, n_classes=6):
        super().__init__()

        # verschillende lagen van neurale netwerk
        self.network = nn.Sequential(

            # 1e verborgen laag: #inputfeatures --> 128 neuronen
            nn.Linear(input_size, 128),

            # Activatiefunctie --> leer niet-lineaire verbanden 
            nn.ReLU(),

            # 20% uitschakelen tegen overfitting
            nn.Dropout(0.2),

            # Tweede verborgen laag:128 --> 64 neuronen
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Outputlaag: 1 uitvoerwaarde per klasse
            nn.Linear(64, n_classes))

    def forward(self, x):

        # Geeft de input door alle lagen
        return self.network(x)

In [10]:
# Traininsfunctie 

def soft_cross_entropy(logits, targets):

    # Zet de ruwe modeluitvoer om naar log-kansen
    log_probs = torch.log_softmax(logits, dim=1)

    # Vergelijk de voorspelde kansen met de echte soft labels
    loss = -(targets * log_probs).sum(dim=1).mean()

    return loss


def train_mlp(model, train_loader, epochs=100):

    # Adam = standaard optimalisatiemethode
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=0.0001)

    # trainingsmodus
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        # Verwerk de trainingsgegevens per batch
        for batch_X, batch_Y in train_loader:

            # Wis de gradiënten van vorige stap
            optimizer.zero_grad()

            # Laat het model voorspellingen maken
            logits = model(batch_X)

            # Bereken hoe groot voorspelfout 
            loss = soft_cross_entropy(logits, batch_Y)

            # Bereken hoe gewichten aanpassen 
            loss.backward()

            # Gewichten aanpassen 
            optimizer.step()

            total_loss += loss.item()

        # Toon om de 10 epochs 
        if (epoch + 1) % 10 == 0:
            average_loss = total_loss / len(train_loader)

            print(
                f"Epoch {epoch + 1}/{epochs} | "
                f"Training loss: {average_loss:.5f}"
            )

    return model

In [11]:
# cross-validatie 

# Instellingen
N_FOLDS = 3
EPOCHS = 100
BATCH_SIZE = 128
SEED = 42

# reproduceerbaarheid
torch.manual_seed(SEED)
np.random.seed(SEED)

# Gebruik GPU indien beschikbaar
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")
print("Gebruikt apparaat:", DEVICE)

# Controleer of gegevens klaarstaan
assert X.shape[0] == Y.shape[0]
assert len(groups) == len(X)

# Bewaar voorspellingen
mlp_oof = np.zeros_like(Y, dtype=np.float64)

# Maak de drie folds
gkf = GroupKFold(n_splits=N_FOLDS)

for fold, (tr, va) in enumerate(gkf.split(X, Y, groups), start=1):

    print(f"\n========== FOLD {fold} ==========")

    # Schaal de features op basis van de trainingsgegevens
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[tr]).astype(np.float32)
    X_valid = scaler.transform(X[va]).astype(np.float32)
    Y_train = Y[tr].astype(np.float32)

    # Gegevens omzetten naar PyTorch-tensors
    X_train_tensor = torch.tensor(X_train)
    Y_train_tensor = torch.tensor(Y_train)

    X_valid_tensor = torch.tensor(
        X_valid,
        dtype=torch.float32,
        device=DEVICE)

    # Maak batches van de trainingsgegevens
    train_dataset = TensorDataset(
        X_train_tensor,
        Y_train_tensor)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True)

    # Maak een nieuw MLP voor deze fold
    model = MLPClassifier(
        input_size=X.shape[1],
        n_classes=Y.shape[1]).to(DEVICE)

    # Train het model
    model = train_mlp(
        model,
        train_loader,
        epochs=EPOCHS)

    # evaluatiemodus
    model.eval()

    # Voorspel validatiegegevens
    with torch.no_grad():
        logits = model(X_valid_tensor)

        # Uitvoer in kansen 
        predictions = torch.softmax(logits, dim=1)
        predictions = predictions.cpu().numpy()

    # Bewaar de voorspellingen
    mlp_oof[va] = predictions

    # Bereken de KL-score
    fold_score = baseline.kl_score(
        Y[va],
        predictions)

    print(
        f"Fold {fold} | "
        f"KL-score: {fold_score:.5f}")

    print(f"Fold {fold} afgerond.")

Gebruikt apparaat: cpu

========== FOLD 1 ==========
Epoch 10/100 | Training loss: 1.25124
Epoch 20/100 | Training loss: 1.15518
Epoch 30/100 | Training loss: 1.09986
Epoch 40/100 | Training loss: 1.05812
Epoch 50/100 | Training loss: 1.03568
Epoch 60/100 | Training loss: 1.00527
Epoch 70/100 | Training loss: 0.98418
Epoch 80/100 | Training loss: 0.96237
Epoch 90/100 | Training loss: 0.93467
Epoch 100/100 | Training loss: 0.92717
Fold 1 | KL-score: 1.08283
Fold 1 afgerond.

========== FOLD 2 ==========
Epoch 10/100 | Training loss: 1.21000
Epoch 20/100 | Training loss: 1.13251
Epoch 30/100 | Training loss: 1.07257
Epoch 40/100 | Training loss: 1.04128
Epoch 50/100 | Training loss: 1.00385
Epoch 60/100 | Training loss: 0.97927
Epoch 70/100 | Training loss: 0.96462
Epoch 80/100 | Training loss: 0.94365
Epoch 90/100 | Training loss: 0.92704
Epoch 100/100 | Training loss: 0.91755
Fold 2 | KL-score: 1.25627
Fold 2 afgerond.

========== FOLD 3 ==========
Epoch 10/100 | Training loss: 1.20870

In [12]:
# Resultaten bekijken 

# totale KL-score
mlp_score = baseline.kl_score(Y, mlp_oof)

# overzicht
mlp_results = pd.DataFrame([{
    "Experiment": "MLP",
    "Model": "Multi-Layer Perceptron",
    "KL_score": mlp_score,
    "Status": "Completed"}])

print("========== MLP RESULTAAT ==========")
print(mlp_results.to_string(index=False))


========== MLP RESULTAAT ==========
Experiment                  Model  KL_score    Status
       MLP Multi-Layer Perceptron  1.159326 Completed
